# Deliverable 2 - INF201 - Example solution


In [46]:
# Import modules used for this assignment
import json
import re
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Task 1: File ingestion and regular expressions

### Function for reading log files

In [47]:
def read_raw_log(file_path: Path | str) -> tuple[str, str]:
    """Reads file content using UTF-8, falling back to ISO-8859-1."""
    path = Path(file_path)
    try:
        return path.read_text(encoding="utf-8"), "UTF-8"
    except UnicodeDecodeError:
        return path.read_text(encoding="iso-8859-1"), "ISO-8859-1"

### Function for parsing logs using regular expressions

In [48]:
def parse_log_lines(raw_text: str) -> list[dict[str, Any]]:
    """Extracts valid sales transaction records from raw log text."""
    pattern = re.compile(
        r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
        r"\[(?P<register_id>[A-Za-z0-9_]+)\]\s+"
        r"item=(?P<item>[A-Za-z0-9_]+)\s+"
        r"price=(?P<price>-?\d+(?:\.\d+)?)\s+"
        r"(?P<currency>[A-Za-z€]+)\s+"
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
    )

    records = []
    for line in raw_text.splitlines():
        match = pattern.match(line.strip())
        if match:
            data = match.groupdict()
            records.append({
                "timestamp": data["timestamp"],
                "register_id": data["register_id"],
                "item": data["item"],
                "price": float(data["price"]),
                "currency": data["currency"],
                "status": data["status"],
            })
    return records

## Task 2: Recursive utility functions

### Recursive file finder

In [49]:
def find_file_paths(directory_path: Path | str, extension: str = ".log") -> list[Path]:
    """Recursively collects file paths matching extension using pathlib."""
    dir_path = Path(directory_path)
    found_files = []

    if not dir_path.is_dir():
        return found_files

    for item in dir_path.iterdir():
        if item.is_dir():
            found_files.extend(find_file_paths(item, extension))
        elif item.suffix == extension:
            found_files.append(item)

    return found_files

### Recursive metadata extraction

In [50]:
def extract_active_registers(config: dict[str, Any] | list[Any] | Any) -> set[str]:
    """Recursively gathers IDs of all active cash registers."""
    active_ids = set()

    if isinstance(config, dict):
        if config.get("active") is True and "id" in config:
            active_ids.add(str(config["id"]))
        for value in config.values():
            active_ids.update(extract_active_registers(value))

    elif isinstance(config, list):
        for item in config:
            active_ids.update(extract_active_registers(item))

    return active_ids

## Task 3: Functions for cleaning and analysing data using NumPy

### Function for removing price outliers

In [51]:
def clean_prices(prices: np.ndarray) -> np.ndarray:
    """Replaces price outliers (> 3 std dev) with the array median."""
    arr = np.array(prices, dtype=float, copy=True)
    if arr.size == 0:
        return arr

    mean = np.mean(arr)
    std = np.std(arr)
    if std == 0:
        return arr

    outlier_mask = np.abs(arr - mean) > (3 * std)
    median_val = np.median(arr[~outlier_mask])
    arr[outlier_mask] = median_val
    return arr

### Computing refund ratios

In [52]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Calculates vectorized refund ratio with division safety."""
    return refund_count / (sales_count + 1e-6)

## Task 4: Data analysis using Pandas

### Build DataFrame

In [53]:
def build_dataframe(records: list[dict[str, Any]]) -> pd.DataFrame:
    """Converts parsed records into a DataFrame and sets timestamps as index."""
    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)

    # Set timestamp index
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df.set_index("timestamp", inplace=True)
    df.sort_index(inplace=True)

    return df

### Resample hourly revenue

In [54]:
def resample_hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Resamples transactions into 1-hour total sales summaries per item."""
    if df.empty:
        return df

    return (
        df.groupby(["item"])["price"]
        .resample("1h")
        .sum()
        .reset_index()
    )

### Status pivot table

In [55]:
def create_status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Creates pivot table showing total sales volume per item and status."""
    if df.empty:
        return pd.DataFrame()

    return pd.pivot_table(
        df,
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0.0,
    )

### Refund ratio table

In [56]:
def calculate_item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates the refund ratio per item by passing transaction count
    arrays into the compute_refund_ratio function.
    """
    if df.empty:
        return pd.DataFrame()

    # Count occurrences of each status per item
    counts = df.groupby(["item", "status"]).size().unstack(fill_value=0)

    # Ensure required columns exist even if zero transactions occurred
    for status_col in ["SUCCESS", "REFUND"]:
        if status_col not in counts.columns:
            counts[status_col] = 0

    # Extract transaction counts as 1D NumPy arrays
    sales_counts = counts["SUCCESS"].to_numpy()
    refund_counts = counts["REFUND"].to_numpy()

    # Compute vectorized refund ratio
    ratios = compute_refund_ratio(sales_counts, refund_counts)

    # Attach ratio back to summary DataFrame
    counts["refund_ratio"] = np.round(ratios, 4)

    return counts[["SUCCESS", "REFUND", "refund_ratio"]]

## Task 5: Create full workflow

### Helper function for correcting NOK prices

In [57]:
def correct_nok_prices(df: pd.DataFrame) -> pd.DataFrame:
    """Corrects wrongly registered NOK prices by multiplying by 10."""
    df_clean = df.copy()
    nok_mask = df_clean["currency"] == "NOK"
    df_clean.loc[nok_mask, "price"] = (df_clean.loc[nok_mask, "price"] * 10.0).round(2)
    return df_clean

### Process data and write output files

In [58]:

def process_data(data_dir: Path, output_dir: Path) -> None:
    """
    Process log files and configuration file in the nested directory data_dir,
    and write result files to output_dir
    """

    # Create output directory if it does not already exist
    output_dir.mkdir(exist_ok=True)

    # Check whether data_dir exist, and return if not
    if not data_dir.exists():
        print("Error: 'data' directory missing. Please run generate_dataset.py first.")
        return

    # Read the config file to find active register IDs
    print("1. RECURSIVE DISCOVERY")
    config_file = data_dir / "store_config.json"
    if config_file.exists():
        config_data = json.loads(config_file.read_text(encoding="utf-8"))
        active_registers = extract_active_registers(config_data)
        print(f"   Active Register IDs: {active_registers}")

    # Find all log files in the nested directory
    log_files = find_file_paths(data_dir / "2026", extension=".log")
    print(f"   Found {len(log_files)} log files.")

    # Extract data from the log files using regular expressions
    # and collect them in the list all_records
    print("\n2. INGESTION & REGEX PARSING")
    all_records = []
    for log_path in log_files:
        raw_text, _ = read_raw_log(log_path)
        records = parse_log_lines(raw_text)
        all_records.extend(records)
    print(f"   Parsed {len(all_records)} valid transactions.")

    print("\n3. DATA CLEANING & CURRENCY ADJUSTMENT")
    # Convert parsed records to DataFrame
    df = build_dataframe(all_records)

    # Clean extreme glitch outliers (999.00)
    # Note: DO this before converting the NOK prices, so they are not mistaken as outliers
    raw_prices = df["price"].to_numpy()
    df["price"] = clean_prices(raw_prices)
    print("   ✓ Removed 999.00 price glitches")

    # Scale NOK prices by 10 now that glitches are removed
    df = correct_nok_prices(df)
    print("   ✓ Multiplied NOK prices by 10")

    # Create hourly resampled data, pivot table and refund ratio table
    print("\n4. PANDAS ANALYTICS & EXPORT")
    hourly_df = resample_hourly_revenue(df)
    pivot_df = create_status_pivot(df)
    refund_df = calculate_item_refund_ratios(df)

    print("\nHourly Sales Revenue Summary (Head):")
    print(hourly_df.head())

    print("\nStatus Pivot Table:")
    print(pivot_df)

    print("\nRefund Ratio Table:")
    print(refund_df)

    # Save outputs
    df.to_csv(output_dir / "cleaned_transactions.csv", index=False)
    pivot_df.to_csv(output_dir / "pivotstatus_pivot_table.csv", index=False)
    refund_df.to_csv(output_dir / "item_refund_ratios.csv", index=False)
    hourly_df.to_json(output_dir / "hourly_revenue.json", orient="records", indent=2, date_format="iso")
    print(f"\nSaved output files to './{output_dir}'. Complete!")


# Define directories and run the analysis
my_data_dir = Path("C:\\", "Users", "tutor5923", "Documents", "folder_for_saving", "d2_data")  # Path("data"))
my_output_dir = Path("output")
process_data(my_data_dir, my_output_dir)

1. RECURSIVE DISCOVERY
   Active Register IDs: {'POS_01', 'POS_02', 'POS_03'}
   Found 9 log files.

2. INGESTION & REGEX PARSING
   Parsed 720 valid transactions.

3. DATA CLEANING & CURRENCY ADJUSTMENT
   ✓ Removed 999.00 price glitches
   ✓ Multiplied NOK prices by 10

4. PANDAS ANALYTICS & EXPORT

Hourly Sales Revenue Summary (Head):
         item           timestamp    price
0  Cappuccino 2026-10-15 07:00:00  1066.26
1  Cappuccino 2026-10-15 08:00:00   702.93
2  Cappuccino 2026-10-15 09:00:00   142.06
3   Croissant 2026-10-15 07:00:00  1228.51
4   Croissant 2026-10-15 08:00:00   789.77

Status Pivot Table:
status      CANCELLED  REFUND  SUCCESS
item                                  
Cappuccino     346.07  285.95  1279.23
Croissant      498.26  278.71  1310.15
Espresso       288.35  305.44  1001.94
Latte          397.98  733.38  1323.25
Muffin         152.24  242.18  1234.76
Sandwich       749.35  274.08  1799.82

Refund Ratio Table:
status      SUCCESS  REFUND  refund_ratio
item  